In [1]:
from treeanalysis import TreeAnalysis
import polars as pl

### set folder and file paths

In [2]:
folder_path = "/Users/hasss/dev/bol.com/modelSnapshots/"
model_filename = "Data-Decision-ADM-ModelSnapshot_AdmModelsSnapshot_20230308T113755_GMT/data.json"
plots_folder_path = "/Users/hasss/dev/pega-datascientist-tools/output"
tree_analysis = TreeAnalysis(
    past_n_snapshots='all', 
    compare_first_n_trees=4, 
    folder_path=folder_path,
    model_filename=model_filename,
    plots_folder_path=plots_folder_path
)

In [3]:
df = tree_analysis.get_df()

AGB models found: shape: (2, 1)
┌───────────────────────────────┐
│ Configuration                 │
│ ---                           │
│ cat                           │
╞═══════════════════════════════╡
│ Mobile_Click_Through_Rate_AGB │
│ Web_Click_Through_Rate_AGB    │
└───────────────────────────────┘


100%|██████████| 13/13 [00:09<00:00,  1.41it/s]


Comparing snapshots 2023-03-02 06:00:02 and 2023-03-03 06:00:01
Comparing snapshots 2023-03-03 06:00:01 and 2023-03-04 06:00:01
Comparing snapshots 2023-03-04 06:00:01 and 2023-03-05 06:00:01
Comparing snapshots 2023-03-05 06:00:01 and 2023-03-06 06:00:01
Comparing snapshots 2023-03-06 06:00:01 and 2023-03-07 06:00:01
Comparing snapshots 2023-03-07 06:00:01 and 2023-03-08 06:00:01
Comparing snapshots 2023-03-02 06:00:02 and 2023-03-03 06:00:01
Comparing snapshots 2023-03-03 06:00:01 and 2023-03-04 06:00:01
Comparing snapshots 2023-03-04 06:00:01 and 2023-03-05 06:00:01
Comparing snapshots 2023-03-05 06:00:01 and 2023-03-06 06:00:01
Comparing snapshots 2023-03-06 06:00:01 and 2023-03-07 06:00:01
Comparing snapshots 2023-03-07 06:00:01 and 2023-03-08 06:00:01


In [32]:
df.head().sort(["channel", "snapshot_from"])

channel,snapshot_from,snapshot_to,snapshot,tree_id,change_type,node,predictor,score,parent_node,gain,split,left_child,right_child,depth,flag
str,str,str,str,str,str,str,str,f64,f64,f64,str,f64,f64,i64,str
"""Mobile_Click_T...","""20230303""","""20230304""","""20230303""","""tree1""","""identical""","""node1""","""IH.Web.Inbound...",-0.196341,null,566.330154,"""IH.Web.Inbound...",2.0,19.0,1,"""white"""
"""Mobile_Click_T...","""20230303""","""20230304""","""20230303""","""tree1""","""identical""","""node2""","""IH.Web.Inbound...",-0.197197,null,278.575142,"""IH.Web.Inbound...",2.0,19.0,1,"""white"""
"""Mobile_Click_T...","""20230303""","""20230304""","""20230303""","""tree1""","""identical""","""node1""","""Customer.Behav...",-0.196758,1.0,59.753127,"""Customer.Behav...",3.0,8.0,2,"""white"""
"""Mobile_Click_T...","""20230303""","""20230304""","""20230303""","""tree1""","""identical""","""node2""","""Customer.Behav...",-0.198063,1.0,80.276796,"""Customer.Behav...",3.0,10.0,2,"""white"""
"""Mobile_Click_T...","""20230303""","""20230304""","""20230303""","""tree1""","""identical""","""node1""","""IH.Web.Inbound...",-0.182016,2.0,7.189386,"""IH.Web.Inbound...",9.0,16.0,3,"""white"""


In [7]:
from enum import Enum
CHANGE_TYPES = Enum('ChangeType', ['split_to_prune', 'split_to_split', 'prune_to_split', 'identical'])
print(CHANGE_TYPES.split_to_prune.name)

NODES = Enum('Nodes', ['node1', 'node2'])
print(NODES.node1.name)

split_to_prune
node1


In [6]:
def run_query(query):
    with pl.Config() as cfg:
        cfg.set_fmt_str_lengths(100)
        display(query)

In [8]:

query = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.split_to_prune.name) &
        (pl.col("node") == NODES.node1.name)
    )
    .groupby(['channel', 'change_type', 'predictor'])
    .agg(pl.count())
    .select(pl.col('channel', 'change_type', 'predictor', 'count').sort_by("channel", "predictor"))
)
run_query(query)

channel,change_type,predictor,count
str,str,str,u32
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.BehavioralData.L1ShelfVisitMostInSession""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.BehavioralData.VisitShareL1Games""",2
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.BehavioralData.VisitShareL1Persoonlijkeverzorging""",2
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.DaysSinceLastOrder""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.Is13261CategoryOrderedIn24M""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.NextBestShelfL2""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""Customer.OrderShareDeviceMobile""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""IH.Web.Inbound.Clicked.pxLastGroupID""",1
"""Mobile_Click_Through_Rate_AGB""","""split_to_prune""","""IH.Web.Inbound.Clicked.pxLastOutcomeTime.DaysSince""",1


In [9]:
item = query[2,:]
display(item)

query1 = (
    df.filter(
            (pl.col("node") == NODES.node1.name) &
            (pl.col("channel") == item.select("channel")) &
            (pl.col("change_type") == item.select("change_type")) & 
            (pl.col("predictor") == item.select("predictor"))
        )
)
run_query(query1)

channel,change_type,predictor,count
str,str,str,u32
"""Mobile_Click_T...","""split_to_prune...","""Customer.Behav...",2


channel,snapshot_from,snapshot_to,snapshot,tree_id,change_type,node,predictor,score,parent_node,gain,split,left_child,right_child,depth,flag
str,str,str,str,str,str,str,str,f64,f64,f64,str,f64,f64,i64,str
"""Mobile_Click_Through_Rate_AGB""","""20230306""","""20230307""","""20230306""","""tree4""","""split_to_prune""","""node1""","""Customer.BehavioralData.VisitShareL1Persoonlijkeverzorging""",-0.133338,30.0,0.616041,"""Customer.BehavioralData.VisitShareL1Persoonlijkeverzorging < 0.1481""",35.0,36.0,4,"""brown1"""
"""Mobile_Click_Through_Rate_AGB""","""20230305""","""20230306""","""20230305""","""tree3""","""split_to_prune""","""node1""","""Customer.BehavioralData.VisitShareL1Persoonlijkeverzorging""",-0.14528,8.0,0.672394,"""Customer.BehavioralData.VisitShareL1Persoonlijkeverzorging < 0.1481""",13.0,20.0,4,"""brown1"""


In [10]:
query2 = (
    df
    .filter(pl.col("node") == NODES.node1.name)
    .groupby(['channel', 'change_type', 'predictor', 'depth'])
    .agg(pl.count())
)
run_query(query2)

channel,change_type,predictor,depth,count
str,str,str,i64,u32
"""Web_Click_Through_Rate_AGB""","""identical""","""Param.ClickedCount1Day""",4,2
"""Mobile_Click_Through_Rate_AGB""","""identical""","""IH.Web.Inbound.Clicked.pyHistoricalOutcomeCount""",1,18
"""Web_Click_Through_Rate_AGB""","""identical""","""Customer.BehavioralData.RetailActionPageVisitInSession""",1,6
"""Mobile_Click_Through_Rate_AGB""","""split_to_split""","""Customer.FavouriteConsole""",3,1
"""Mobile_Click_Through_Rate_AGB""","""split_to_split""","""Customer.BehavioralData.SecondLastL2ShelfVisitInSession""",3,1
"""Mobile_Click_Through_Rate_AGB""","""identical""","""Param.ShelfLevel2""",4,1
"""Web_Click_Through_Rate_AGB""","""identical""","""Customer.BehavioralData.VisitShareL1Beauty""",2,1
"""Web_Click_Through_Rate_AGB""","""split_to_split""","""Customer.BehavioralData.SecondLastL2ShelfVisitInSession""",3,1
"""Web_Click_Through_Rate_AGB""","""identical""","""Param.ShelfLevel2""",3,2


#### query:: average gain of splits that were pruned

In [26]:

query3 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.split_to_prune.name) &
        (pl.col("node") == NODES.node1.name)
    )
    .groupby(['channel'])
    .agg(pl.col("gain").mean())
)
run_query(query3)

channel,gain
str,f64
"""Mobile_Click_Through_Rate_AGB""",10.514008
"""Web_Click_Through_Rate_AGB""",10.514008


#### query:: average gain of identical

In [12]:
query4 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.identical.name) &
        (pl.col("node") == NODES.node1.name)
    )
    .groupby(['channel'])
    .agg(pl.col("gain").mean())
)
run_query(query4)

channel,gain
str,f64
"""Mobile_Click_Through_Rate_AGB""",69.212745
"""Web_Click_Through_Rate_AGB""",69.212745


#### query:: avg gain of prune to split

In [29]:
q1 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.prune_to_split.name) &
        (pl.col("node") == NODES.node1.name) &
        (pl.col("tree_id") == "tree1")
        (pl.col("snapshot") == )
    )
    .groupby(['channel'])
    .agg(pl.col("gain").mean())
)
q2 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.prune_to_split.name) &
        (pl.col("node") == NODES.node2.name) &
        (pl.col("tree_id") == "tree1")
    )
    .groupby(['channel'])
    .agg(pl.col("gain").mean())
    .rename({
        "gain": "gain_"
    })
)


run_query(q1.join(q2, on="channel"))

channel,gain,gain_
str,f64,f64
"""Web_Click_Through_Rate_AGB""",0.0,2.366961
"""Mobile_Click_Through_Rate_AGB""",0.0,2.366961


#### query:: split to split

In [28]:
q11 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.split_to_split.name) &
        (pl.col("node") == NODES.node1.name) &
        (pl.col("tree_id") == "tree1")
    )
    .groupby(['channel'])
    .agg(pl.col("gain").mean())
)
q12 = (
    df
    .filter(
        (pl.col("change_type") == CHANGE_TYPES.split_to_split.name) &
        (pl.col("node") == NODES.node2.name) &
        (pl.col("tree_id") == "tree1")
    )
    .groupby(['channel'])
    .agg(pl.col("gain").mean())
    .rename({
        "gain": "gain_"
    })
)


run_query(q11.join(q12, on="channel"))

channel,gain,gain_
str,f64,f64
"""Web_Click_Through_Rate_AGB""",7.97627,12.412453
"""Mobile_Click_Through_Rate_AGB""",7.97627,12.412453
